In [1]:
import pandas as pd
import numpy as np

# ── Load both datasets ──────────────────────────────────────────
pima = pd.read_csv(r"C:\Users\abdul\Amr senior\dataset\archive (8)\diabetes.csv")
frankfurt = pd.read_csv(r"C:\Users\abdul\Amr senior\dataset\archive (7)\diabetes.csv")

print("Pima shape:      ", pima.shape)
print("Frankfurt shape: ", frankfurt.shape)

# ── Align columns (in case of any naming differences) ───────────
print("\nPima columns:     ", pima.columns.tolist())
print("Frankfurt columns:", frankfurt.columns.tolist())

# Rename to match if needed (usually identical)
frankfurt.columns = pima.columns

# ── Merge ────────────────────────────────────────────────────────
df = pd.concat([pima, frankfurt], ignore_index=True)
print(f"\nMerged shape: {df.shape}")  # Expected: (2768, 9)

# ── Fix zero values (medically impossible zeros → NaN) ──────────
zero_invalid_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[zero_invalid_cols] = df[zero_invalid_cols].replace(0, np.nan)

print("\nMissing values after fixing zeros:")
print(df.isnull().sum())

# ── Impute with median ───────────────────────────────────────────
for col in zero_invalid_cols:
    df[col].fillna(df[col].median(), inplace=True)

# ── Check class balance ──────────────────────────────────────────
print("\nClass distribution:")
print(df['Outcome'].value_counts())
print(df['Outcome'].value_counts(normalize=True).mul(100).round(1))

# ── Save merged dataset ──────────────────────────────────────────
save_path = r"C:\Users\abdul\Amr senior\dataset\diabetes_merged.csv"
df.to_csv(save_path, index=False)
print(f"\nSaved to: {save_path}")

Pima shape:       (768, 9)
Frankfurt shape:  (2000, 9)

Pima columns:      ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
Frankfurt columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

Merged shape: (2768, 9)

Missing values after fixing zeros:
Pregnancies                    0
Glucose                       18
BloodPressure                125
SkinThickness                800
Insulin                     1330
BMI                           39
DiabetesPedigreeFunction       0
Age                            0
Outcome                        0
dtype: int64

Class distribution:
Outcome
0    1816
1     952
Name: count, dtype: int64
Outcome
0    65.6
1    34.4
Name: proportion, dtype: float64

Saved to: C:\Users\abdul\Amr senior\dataset\diabetes_merged.csv


C:\Users\abdul\AppData\Local\Temp\ipykernel_26020\6553557.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\abdul\AppData\Local\Temp\ipykernel_26020\6553557.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, 

In [3]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

df = pd.read_csv(r"C:\Users\abdul\Amr senior\dataset\diabetes_merged.csv")

# ── Step 1: Smart Imputation (KNN-style iterative) ───────────────
# Much better than median for high-missing columns like Insulin
imputer = IterativeImputer(max_iter=10, random_state=42)
cols_to_impute = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']

df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])

# Clip negative values that imputer might produce
df[cols_to_impute] = df[cols_to_impute].clip(lower=0)

print("✅ Missing after imputation:")
print(df.isnull().sum().sum(), "nulls remaining")

# ── Step 2: Feature Engineering ──────────────────────────────────
df['Glucose_BMI']    = df['Glucose'] * df['BMI']
df['Age_BMI']        = df['Age'] * df['BMI']
df['Glucose_squared']= df['Glucose'] ** 2
df['Insulin_BMI']    = df['Insulin'] * df['BMI']

print("\n✅ Features after engineering:", df.shape[1] - 1, "features")

# ── Step 3: Split BEFORE scaling/SMOTE (prevent data leakage) ────
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Step 4: Normalize ─────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit only on train
X_test_scaled  = scaler.transform(X_test)         # transform test

# ── Step 5: SMOTE on training set only ───────────────────────────
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

print("\n✅ Class distribution after SMOTE:")
print(pd.Series(y_train_bal).value_counts())

# ── Step 6: Save processed splits ────────────────────────────────
base = r"C:\Users\abdul\Amr senior\dataset"
np.save(f"{base}\\X_train.npy", X_train_bal)
np.save(f"{base}\\X_test.npy",  X_test_scaled)
np.save(f"{base}\\y_train.npy", y_train_bal)
np.save(f"{base}\\y_test.npy",  y_test.values)
import joblib
joblib.dump(scaler, f"{base}\\scaler.pkl")

print("\n✅ All files saved and ready for training!")
print(f"   Train: {X_train_bal.shape} | Test: {X_test_scaled.shape}")

✅ Missing after imputation:
0 nulls remaining

✅ Features after engineering: 12 features

✅ Class distribution after SMOTE:
Outcome
0    1453
1    1453
Name: count, dtype: int64

✅ All files saved and ready for training!
   Train: (2906, 12) | Test: (554, 12)
